In [ ]:
import os
import re
import json
import csv
from bs4 import BeautifulSoup
from urllib.parse import urljoin

BASE_URL = "https://www.geneon.net"

def clean_text(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip()

def extract_url(soup: BeautifulSoup) -> str:
    # Best: canonical URL
    canon = soup.find("link", rel="canonical")
    if canon and canon.get("href"):
        return canon["href"].strip()

    # Backup: og:url
    og = soup.find("meta", property="og:url")
    if og and og.get("content"):
        return og["content"].strip()

    return ""

def get_main_container(soup: BeautifulSoup):
    # GeneON pages usually have this
    main = soup.find(id="content_area")
    return main if main else soup.body

def extract_texts(container) -> str:
    if not container:
        return ""
    # remove junk
    for tag in container.find_all(["script", "style", "noscript"]):
        tag.decompose()
    return clean_text(container.get_text(" ", strip=True))

def extract_price(soup: BeautifulSoup) -> str:
    # Strong signal: schema.org Offer price
    price_tag = soup.find(attrs={"itemprop": "price"})
    if price_tag:
        return clean_text(price_tag.get_text(" ", strip=True))

    # Backup: find something like "290,00 €"
    text = soup.get_text(" ", strip=True)
    m = re.search(r"\b\d{1,3}(?:\.\d{3})*,\d{2}\s*€\b", text)
    return m.group(0) if m else ""

def extract_images(container) -> list:
    imgs = []
    if not container:
        return imgs

    for img in container.find_all("img"):
        src = img.get("src") or img.get("data-src")
        if not src:
            continue
        # filter obvious icons
        low = src.lower()
        if "icons" in low or "pdf.png" in low:
            continue
        imgs.append(src)

    # unique, keep order
    seen = set()
    out = []
    for u in imgs:
        if u not in seen:
            seen.add(u)
            out.append(u)
    return out

def extract_datasheets(container) -> list:
    links = []
    if not container:
        return links

    for a in container.find_all("a", href=True):
        href = a["href"].strip()
        low = href.lower()
        if ".pdf" in low or "/app/download/" in low:
            links.append(urljoin(BASE_URL, href))

    # unique
    return list(dict.fromkeys(links))

def extract_article_number(soup: BeautifulSoup) -> str:
    """
    Many GeneON pages (Jimdo shop) have product container IDs like:
    cc-m-product-14251324896
    We'll extract that numeric ID as a stable "article number" IF no other is visible.
    """
    prod = soup.find(attrs={"itemscope": True, "itemtype": re.compile("schema.org/Product", re.I)})
    if prod and prod.get("id"):
        m = re.search(r"(\d+)", prod["id"])
        if m:
            return m.group(1)
    return ""

def extract_variants(container) -> list:
    """
    Variants often appear as <select> dropdowns or option groups.
    If none exist, return [].
    """
    variants = []
    if not container:
        return variants

    # check select dropdowns
    for sel in container.find_all("select"):
        opts = [clean_text(o.get_text(" ", strip=True)) for o in sel.find_all("option")]
        opts = [o for o in opts if o]
        if len(opts) > 1:
            variants.append(opts)

    return variants

def parse_file(html_path: str) -> dict:
    with open(html_path, "r", encoding="utf-8", errors="ignore") as f:
        soup = BeautifulSoup(f.read(), "html.parser")

    container = get_main_container(soup)

    return {
        "file": os.path.basename(html_path),
        "url": extract_url(soup),
        "texts": extract_texts(container),
        "price": extract_price(soup),
        "images": extract_images(container),
        "article_number": extract_article_number(soup),
        "datasheets": extract_datasheets(container),
        "variants": extract_variants(container),
    }

def run_folder(input_folder: str, out_json: str = "extracted.json", out_csv: str = "extracted.csv"):
    rows = []
    for name in os.listdir(input_folder):
        if name.lower().endswith(".html"):
            path = os.path.join(input_folder, name)
            rows.append(parse_file(path))

    # Save JSON
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False, indent=2)

    # Save CSV (lists will be stored as JSON strings)
    fieldnames = ["file", "url", "texts", "price", "images", "article_number", "datasheets", "variants"]
    with open(out_csv, "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        for r in rows:
            r2 = r.copy()
            for k in ["images", "datasheets", "variants"]:
                r2[k] = json.dumps(r2[k], ensure_ascii=False)
            w.writerow(r2)

    print(f"✅ Parsed {len(rows)} HTML files")
    print(f"✅ Saved: {out_json}, {out_csv}")

# Example usage:
run_folder("HTML-geneon_pages")

✅ Parsed 443 HTML files
✅ Saved: extracted.json, extracted.csv


In [4]:
import pandas as pd

In [5]:
df = pd.read_csv("extracted.csv")

In [10]:
df.columns
df.dtypes

file               object
url                object
texts              object
price              object
images             object
article_number    float64
datasheets         object
variants           object
dtype: object

In [13]:
df

,file,url,texts,price,images,article_number,datasheets,variants
0,page_0.html,https://www.geneon.net/,Edit here your header slider GeneON-BioScience...,NaN,"[""https://u.jimcdn.com/e/o/s992ea961e2602c7c/u...",NaN,[],[]
1,page_1.html,https://www.geneon.net/english-pages/ablage/,Expression of actin gene with Universal Flex q...,NaN,"[""https://image.jimcdn.com/app/cms/image/trans...",NaN,[],[]
2,page_10.html,https://www.geneon.net/products/dna-rna-extrac...,Blood DNA Purification Kit - extraction of DNA...,"85,00 €","[""https://image.jimcdn.com/app/cms/image/trans...",1.427545e+10,"[""https://www.geneon.net/app/download/18410301...","[[""BD50 - 50 preps 85,00 €"", ""BD100 - 100 prep..."
3,page_100.html,https://www.geneon.net/products/biochemicals-a...,"Buffer for Electrophoresis, Western Blotting, ...","259,00 €","[""https://image.jimcdn.com/app/cms/image/trans...",1.710563e+10,[],[]
4,page_101.html,https://www.geneon.net/agarose/,Agarose for perfect results in gel electrophor...,"179,00 €","[""https://image.jimcdn.com/app/cms/image/trans...",1.522236e+10,"[""https://www.geneon.net/app/download/15629792...","[[""604-005 - 500 gr - Agarose - 179,00 €"", ""60..."
...,...,...,...,...,...,...,...,...
438,page_95.html,https://www.geneon.net/reverse-transcriptase/r...,RNase Inhibitor - RNA protector - RNasin. Avai...,"139,00 €","[""https://image.jimcdn.com/app/cms/image/trans...",1.510945e+10,"[""https://www.geneon.net/app/download/18138666...","[[""105-350 - 10.000 Units - SPECIAL PRICE 139,..."
439,page_96.html,https://www.geneon.net/products/rt-pcr-reverse...,Random Primers Mixture of single-stranded rand...,"28,00 €","[""https://image.jimcdn.com/app/cms/image/trans...",1.510953e+10,"[""https://www.geneon.net/app/download/14409566...",[]
440,page_97.html,https://www.geneon.net/products/rt-pcr-reverse...,RT-PCR Reverse Transcription: Datasheet collec...,NaN,[],NaN,"[""https://www.geneon.net/app/download/15229236...",[]
441,page_98.html,https://www.geneon.net/products/biochemicals-a...,"Agarose, Proteinase K, DNase, RNase A, Lyophil...",NaN,"[""https://image.jimcdn.com/app/cms/image/trans...",NaN,[],[]


In [14]:
import pandas as pd
import ast

df = pd.read_csv("extracted.csv")

# remove pages without price (non-product pages)
df = df[df["price"].notna()]

# convert article number
df["article_number"] = df["article_number"].astype(str)

# convert lists stored as text
df["images"] = df["images"].apply(ast.literal_eval)
df["datasheets"] = df["datasheets"].apply(ast.literal_eval)
df["variants"] = df["variants"].apply(ast.literal_eval)

df.to_csv("clean_geneon_products.csv", index=False)

print("Clean products:", len(df))

Clean products: 370


In [15]:
df

,file,url,texts,price,images,article_number,datasheets,variants
2,page_10.html,https://www.geneon.net/products/dna-rna-extrac...,Blood DNA Purification Kit - extraction of DNA...,"85,00 €",[https://image.jimcdn.com/app/cms/image/transf...,14275448096.0,[https://www.geneon.net/app/download/184103013...,"[[BD50 - 50 preps 85,00 €, BD100 - 100 preps 1..."
3,page_100.html,https://www.geneon.net/products/biochemicals-a...,"Buffer for Electrophoresis, Western Blotting, ...","259,00 €",[https://image.jimcdn.com/app/cms/image/transf...,17105627996.0,[],[]
4,page_101.html,https://www.geneon.net/agarose/,Agarose for perfect results in gel electrophor...,"179,00 €",[https://image.jimcdn.com/app/cms/image/transf...,15222360796.0,[https://www.geneon.net/app/download/156297922...,"[[604-005 - 500 gr - Agarose - 179,00 €, 604-0..."
5,page_102.html,https://www.geneon.net/products/enzymes-and-fi...,Low Melt Agarose (PRODUCT IS DISCONTINUED) Low...,"0,00 €",[https://image.jimcdn.com/app/cms/image/transf...,15222356696.0,[https://www.geneon.net/app/download/152290976...,[]
6,page_103.html,https://www.geneon.net/bsa-bovine-serum-albumin/,Bovine Serum Albumin - BSA nativ - non-acetyla...,"38,00 €",[https://image.jimcdn.com/app/cms/image/transf...,15222353296.0,[https://www.geneon.net/app/download/152290920...,"[[209-005 (Glycerol NEB-Buffer) - 4x1,5 ml - 3..."
...,...,...,...,...,...,...,...,...
435,page_92.html,https://www.geneon.net/reverse-transcription/m...,M-MuLV Reverse Transcriptase M-MuLV-Reverse Tr...,"229,00 €",[https://image.jimcdn.com/app/cms/image/transf...,15109423896.0,[https://www.geneon.net/app/download/164805513...,"[[105-250- 50.000 Units - 229,00 €, 105-100 - ..."
436,page_93.html,https://www.geneon.net/products/rt-pcr-reverse...,AMV Reverse Transcriptase AMV Reverse Transcri...,"0,00 €",[https://image.jimcdn.com/app/cms/image/transf...,15109570196.0,[https://www.geneon.net/app/download/152292365...,[]
437,page_94.html,https://www.geneon.net/products/rt-pcr-reverse...,Oligo (dT)15 Standard primer for RT-PCR Descri...,"31,00 €",[https://image.jimcdn.com/app/cms/image/transf...,15109512996.0,[https://www.geneon.net/app/download/152292393...,[]
438,page_95.html,https://www.geneon.net/reverse-transcriptase/r...,RNase Inhibitor - RNA protector - RNasin. Avai...,"139,00 €",[https://image.jimcdn.com/app/cms/image/transf...,15109450596.0,[https://www.geneon.net/app/download/181386660...,"[[105-350 - 10.000 Units - SPECIAL PRICE 139,0..."


In [16]:
import pandas as pd
import ast

df = pd.read_csv("extracted.csv")

# remove pages without price
df = df[df["price"].notna()]

# fix article numbers
df["article_number"] = df["article_number"].astype(str)

def safe_eval(x):
    try:
        return ast.literal_eval(x)
    except:
        return []

df["images"] = df["images"].apply(safe_eval)
df["datasheets"] = df["datasheets"].apply(safe_eval)
df["variants"] = df["variants"].apply(safe_eval)

df.to_csv("Clean_geneon_products.csv", index=False)

print("Clean products:", len(df))

Clean products: 370


In [17]:
import pandas as pd

df = pd.read_csv("clean_geneon_products.csv")

# fix article numbers
df["article_number"] = df["article_number"].apply(lambda x: str(int(float(x))) if pd.notna(x) else "")
df = df[df["variants"].astype(str) != "[]"]
df.to_csv("clean_geneon_products_FIXED.csv", index=False)

print("Done")

Done


In [1]:
import os
import re
import json
import csv
from bs4 import BeautifulSoup
from urllib.parse import urljoin

BASE_URL = "https://www.geneon.net"


def clean_text(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip()


def unique_keep_order(items):
    seen = set()
    result = []
    for item in items:
        if item and item not in seen:
            seen.add(item)
            result.append(item)
    return result


def extract_url(soup: BeautifulSoup) -> str:
    # 1) canonical URL
    canon = soup.find("link", rel="canonical")
    if canon and canon.get("href"):
        return canon["href"].strip()

    # 2) fallback: og:url
    og = soup.find("meta", property="og:url")
    if og and og.get("content"):
        return og["content"].strip()

    return ""


def get_main_container(soup: BeautifulSoup):
    main = soup.find(id="content_area")
    return main if main else soup.body


def extract_texts(container) -> str:
    if not container:
        return ""

    # remove unnecessary tags
    for tag in container.find_all(["script", "style", "noscript"]):
        tag.decompose()

    return clean_text(container.get_text(" ", strip=True))


def extract_price(soup: BeautifulSoup) -> str:
    # best case: schema price
    price_tag = soup.find(attrs={"itemprop": "price"})
    if price_tag:
        return clean_text(price_tag.get_text(" ", strip=True))

    # fallback regex
    text = soup.get_text(" ", strip=True)
    match = re.search(r"\b\d{1,3}(?:\.\d{3})*,\d{2}\s*€\b", text)
    return match.group(0) if match else ""


def extract_images(container) -> list:
    """
    Extract only images having:
    class="cc-shop-product-main-image photo"
    """
    if not container:
        return []

    images = []

    # exact class match through CSS selector
    for img in container.select("img.cc-shop-product-main-image.photo"):
        src = img.get("src") or img.get("data-src")
        if src:
            images.append(urljoin(BASE_URL, src.strip()))

    return unique_keep_order(images)


def extract_datasheets(container) -> list:
    """
    Extract only links having:
    class="cc-m-download-link"
    """
    if not container:
        return []

    links = []

    for a in container.find_all("a", class_="cc-m-download-link", href=True):
        href = a["href"].strip()
        links.append(urljoin(BASE_URL, href))

    return unique_keep_order(links)


def extract_article_number(soup: BeautifulSoup) -> str:
    """
    Example product container id:
    cc-m-product-14251324896
    """
    prod = soup.find(
        attrs={
            "itemscope": True,
            "itemtype": re.compile(r"schema\.org/Product", re.I)
        }
    )

    if prod and prod.get("id"):
        match = re.search(r"(\d+)", prod["id"])
        if match:
            return match.group(1)

    return ""


def extract_variants(container) -> list:
    """
    Extract dropdown/select options as variants.
    """
    if not container:
        return []

    variants = []

    for sel in container.find_all("select"):
        options = [
            clean_text(opt.get_text(" ", strip=True))
            for opt in sel.find_all("option")
        ]
        options = [opt for opt in options if opt]

        if len(options) > 1:
            variants.append(options)

    return variants


def parse_file(html_path: str) -> dict:
    with open(html_path, "r", encoding="utf-8", errors="ignore") as f:
        soup = BeautifulSoup(f.read(), "html.parser")

    container = get_main_container(soup)

    return {
        "file": os.path.basename(html_path),
        "url": extract_url(soup),
        "texts": extract_texts(container),
        "price": extract_price(soup),
        "images": extract_images(container),
        "article_number": extract_article_number(soup),
        "datasheets": extract_datasheets(container),
        "variants": extract_variants(container),
    }


def run_folder(input_folder: str, out_json: str = "extracted.json", out_csv: str = "extracted.csv"):
    rows = []

    for name in os.listdir(input_folder):
        if name.lower().endswith(".html"):
            html_path = os.path.join(input_folder, name)
            rows.append(parse_file(html_path))

    # save JSON
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False, indent=2)

    # save CSV
    fieldnames = [
        "file",
        "url",
        "texts",
        "price",
        "images",
        "article_number",
        "datasheets",
        "variants",
    ]

    with open(out_csv, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

        for row in rows:
            row_copy = row.copy()
            for col in ["images", "datasheets", "variants"]:
                row_copy[col] = json.dumps(row_copy[col], ensure_ascii=False)
            writer.writerow(row_copy)

    print(f"✅ Parsed {len(rows)} HTML files")
    print(f"✅ Saved: {out_json}, {out_csv}")


# Example usage
run_folder("Test2-Copy1")

✅ Parsed 443 HTML files
✅ Saved: extracted.json, extracted.csv


In [3]:
import pandas as pd
df = pd.read_csv("extracted.csv")

In [5]:

df = pd.read_csv("extracted.csv")

# remove pages without price
df = df[df["price"].notna()]

# fix article numbers
df["article_number"] = df["article_number"].astype(str)

def safe_eval(x):
    try:
        return ast.literal_eval(x)
    except:
        return []

df["images"] = df["images"].apply(safe_eval)
df["datasheets"] = df["datasheets"].apply(safe_eval)
df["variants"] = df["variants"].apply(safe_eval)

df.to_csv("Clean_geneon_products.csv", index=False)

print("Clean products:", len(df))

Clean products: 370


In [7]:
df = df[df["images"].notna()]

In [8]:
df

,file,url,texts,price,images,article_number,datasheets,variants
2,page_10.html,https://www.geneon.net/products/dna-rna-extrac...,Blood DNA Purification Kit - extraction of DNA...,"85,00 €",[],14275448096.0,[],[]
3,page_100.html,https://www.geneon.net/products/biochemicals-a...,"Buffer for Electrophoresis, Western Blotting, ...","259,00 €",[],17105627996.0,[],[]
4,page_101.html,https://www.geneon.net/agarose/,Agarose for perfect results in gel electrophor...,"179,00 €",[],15222360796.0,[],[]
5,page_102.html,https://www.geneon.net/products/enzymes-and-fi...,Low Melt Agarose (PRODUCT IS DISCONTINUED) Low...,"0,00 €",[],15222356696.0,[],[]
6,page_103.html,https://www.geneon.net/bsa-bovine-serum-albumin/,Bovine Serum Albumin - BSA nativ - non-acetyla...,"38,00 €",[],15222353296.0,[],[]
...,...,...,...,...,...,...,...,...
435,page_92.html,https://www.geneon.net/reverse-transcription/m...,M-MuLV Reverse Transcriptase M-MuLV-Reverse Tr...,"229,00 €",[],15109423896.0,[],[]
436,page_93.html,https://www.geneon.net/products/rt-pcr-reverse...,AMV Reverse Transcriptase AMV Reverse Transcri...,"0,00 €",[],15109570196.0,[],[]
437,page_94.html,https://www.geneon.net/products/rt-pcr-reverse...,Oligo (dT)15 Standard primer for RT-PCR Descri...,"31,00 €",[],15109512996.0,[],[]
438,page_95.html,https://www.geneon.net/reverse-transcriptase/r...,RNase Inhibitor - RNA protector - RNasin. Avai...,"139,00 €",[],15109450596.0,[],[]


In [2]:
import pandas as pd
import ast

# Load CSV
df = pd.read_csv("extracted.csv")

# -------------------------
# 1. Remove spaces safely
# -------------------------
for col in df.columns:
    df[col] = df[col].map(lambda x: x.strip() if isinstance(x, str) else x)

# -------------------------
# 2. Convert list-like text
# -------------------------
list_columns = ["images", "datasheets", "variants"]

for col in list_columns:
    df[col] = df[col].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith("[") else x
    )

# -------------------------
# 3. Clean text column
# -------------------------
df["texts"] = df["texts"].astype(str).str.replace("\n", " ")

# -------------------------
# 4. Convert lists to string
# (for duplicate checking)
# -------------------------
for col in list_columns:
    df[col] = df[col].astype(str)

# -------------------------
# 5. Remove duplicates
# -------------------------
df = df.drop_duplicates()

# -------------------------
# 6. Reset index
# -------------------------
df = df.reset_index(drop=True)

# -------------------------
# 7. Save clean dataset
# -------------------------
df.to_csv("test1.csv", index=False)

print("✅ Clean dataset saved")

✅ Clean dataset saved


In [3]:
df = df[
    df["images"].notna() &
    df["datasheets"].notna() &
    (df["images"] != "[]") &
    (df["datasheets"] != "[]")
]

In [5]:

df.to_csv("test2.csv", index=False)

In [6]:
import ast

df["images"] = df["images"].apply(
    lambda x: ast.literal_eval(x)[0] if isinstance(x, str) and x.startswith("[") else x
)

In [ ]:
import ast

for col in ["images", "datasheets"]:
    df[col] = df[col].apply(
        lambda x: ast.literal_eval(x)[0] if isinstance(x, str) and x.startswith("[") else x
    )

df.to_csv("test4.csv", index=False)

In [1]:
df

NameError: name 'df' is not defined